# 12 — Great Britain in finer detail

Notebook 11 modeled the strategic network alone. This notebook rebuilds the
same national model on a network **more than twice the size**. **Model size:
141,079 links (~70,000 km) across motorways, trunk roads, ramps and every
primary A-road in Great Britain, ~93,000 trigger-built nodes, and the same 88
sectored zones** — against notebook 11's 64,733 links / ~40,000 km. Same zones,
same demand model, same assignment — what changes is what the network can
represent: realistic alternatives to the motorways, gap-filling in Wales, the
South West and Scotland, and traffic relief on corridors that previously had no
competitor routes.

The detailed network ships in `data/uk_roads_detailed.geojson.gz`
(© OpenStreetMap contributors, ODbL, extracted and topologically noded from the
Overpass API). Nothing is downloaded at run time.

In [1]:
import gzip
import json
import time
import warnings
from pathlib import Path
from tempfile import gettempdir
from uuid import uuid4

import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import LineString, shape

warnings.filterwarnings("ignore")

DATA = Path("data")
with gzip.open(DATA / "uk_roads_detailed.geojson.gz", "rt", encoding="utf-8") as fh:
    roads_gj = json.load(fh)
cities = pd.read_csv(DATA / "uk_cities.csv")
boundary = gpd.GeoDataFrame(geometry=[shape(f["geometry"]) for f in json.load(open(DATA / "gb_boundary.geojson"))["features"]], crs=4326)

roads = gpd.GeoDataFrame(
    [{"cls": f["properties"]["class"], "ref": f["properties"]["ref"]} for f in roads_gj["features"]],
    geometry=[shape(f["geometry"]) for f in roads_gj["features"]], crs=4326)

km = roads.to_crs(27700).length.sum() / 1000
print(f"{len(roads):,} links / {km:,.0f} km; {len(cities)} cities, "
      f"{cities.population.sum() / 1e6:.1f}M residents")
roads.groupby("cls").size()

141,079 links / 65,035 km; 74 cities, 34.7M residents


cls
motorway     3172
primary     66629
ramp        18657
trunk       52621
dtype: int64

In [2]:
# JupyterGIS map helper ------------------------------------------------------
# GISDocument is JupyterGIS' notebook API: it builds a live, QGIS-like map
# document rendered directly in JupyterLab. Layers added from GeoDataFrames
# are converted to GeoJSON on the fly.
#
# add_gdf also translates the declarative symbology into the OpenLayers
# flat-style expressions the current JupyterGIS frontend renders from, so
# colours and line widths show up without touching the symbology panel.
import json

import matplotlib.colors
import matplotlib.pyplot as _plt
from jupytergis import GISDocument
from jupytergis_lab.notebook.symbology import to_symbology_state

OSM_TILES = "https://tile.openstreetmap.org/{z}/{x}/{y}.png"

def new_map(gdf_for_extent=None, zoom=12):
    """Create a GISDocument centred on a layer, with an OpenStreetMap basemap."""
    kwargs = {}
    if gdf_for_extent is not None:
        b = gdf_for_extent.total_bounds  # (minx, miny, maxx, maxy)
        kwargs = {"longitude": (b[0] + b[2]) / 2, "latitude": (b[1] + b[3]) / 2, "zoom": zoom}
    doc = GISDocument(**kwargs)
    doc.add_raster_layer(OSM_TILES, name="OpenStreetMap", attribution="(C) OpenStreetMap contributors", opacity=0.6)
    return doc

def _hex(rgba):
    return matplotlib.colors.to_hex(rgba)

def _ramp_expr(fld, params):
    name, dom = params.get("name", "viridis"), params.get("domain") or [0.0, 1.0]
    cmap = _plt.get_cmap(name)
    if params.get("reverse"):
        cmap = cmap.reversed()
    expr = ["interpolate", ["linear"], ["get", fld]]
    for i in range(7):
        t = i / 6
        expr += [dom[0] + t * (dom[1] - dom[0]), _hex(cmap(t))]
    return expr

def _scalar_expr(fld, params):
    d, r = params["domain"], params["range"]
    return ["interpolate", ["linear"], ["get", fld], d[0], r[0], d[1], r[1]]

def _cat_expr(fld, params, gdf):
    cmap = _plt.get_cmap(params.get("colorRamp", "tab10"))
    vals = list(dict.fromkeys(gdf[fld].dropna()))
    expr = ["match", ["get", fld]]
    for i, v in enumerate(vals):
        expr += [v, _hex(cmap(i % cmap.N))]
    return expr + ["#9ca3af"]

def _flat_style(symbology, gdf):
    """Grammar symbology -> OpenLayers flat-style dict (what the map renders)."""
    state = to_symbology_state(symbology)
    if not state:
        return None
    flat = {}
    for layer in state.get("layers", []):
        for rule in layer.get("rules", []):
            flds = rule.get("fields") or [None]
            for m in rule.get("mappings", []):
                scheme = m["scale"]["scheme"]
                params = m["scale"].get("params", {})
                if scheme == "constant_rgba":
                    val = params["value"]
                    val = _hex([c if c <= 1 else c / 255 for c in val]) if isinstance(val, (list, tuple)) else val
                elif scheme == "constant_num":
                    val = params["value"]
                elif scheme == "colorMap":
                    val = _ramp_expr(flds[0], params)
                elif scheme == "scalar":
                    val = _scalar_expr(flds[0], params)
                elif scheme == "categorical":
                    val = _cat_expr(flds[0], params, gdf)
                else:
                    continue
                for enc in m.get("encodings", []):
                    flat[enc] = val
    if any(k.startswith("circle") for k in flat) and "circle-radius" not in flat:
        flat["circle-radius"] = 5
    if "stroke-color" in flat and "stroke-width" not in flat:
        flat["stroke-width"] = 1.5
    return flat or None

def merge_lines(gdf, tol=0.01):
    """Collapse many lines into a single MultiLineString feature.

    Backdrop and class-display layers do not need per-feature identity, and one
    merged feature is a fraction of the size of tens of thousands of features -
    which keeps national-scale layers inside the notebook sync message limit.
    """
    import geopandas as _gpd
    from shapely.geometry import MultiLineString
    parts = []
    for geom in gdf.geometry.simplify(tol):
        if geom is None or geom.is_empty:
            continue
        parts.extend(geom.geoms if geom.geom_type == "MultiLineString" else [geom])
    return _gpd.GeoDataFrame({"links": [len(parts)]}, geometry=[MultiLineString(parts)], crs=gdf.crs)

def _round_coords(o, nd=5):
    if isinstance(o, (int, float)):
        return round(o, nd)
    if isinstance(o, list):
        return [_round_coords(v, nd) for v in o]
    return o

def add_gdf(doc, gdf, name, symbology=None, **kwargs):
    """Add a GeoDataFrame to the map as a GeoJSON layer, with rendered symbology.

    Coordinates are quantized to ~1 m so even national-scale layers stay well
    under Jupyter's websocket message limit (oversized layers are dropped
    silently by the sync, so this matters more than it looks).
    """
    data = json.loads(gdf.to_json())
    for f in data.get("features", []):
        g = f.get("geometry")
        if g and "coordinates" in g:
            g["coordinates"] = _round_coords(g["coordinates"])
    lid = doc.add_geojson_layer(data=data, name=name,
                                symbology=symbology, **kwargs)
    flat = _flat_style(symbology, gdf)
    if flat:
        layer = doc._layers.get(lid)
        layer["parameters"]["color"] = flat
        doc._layers[lid] = layer
    return lid

## The detailed network

Primary A-roads (green) fill in between the trunk (grey) and motorway (blue) skeleton.

In [3]:
from jupytergis_lab.notebook.symbology import constant

cities_gdf = gpd.GeoDataFrame(cities, geometry=gpd.points_from_xy(cities.lon, cities.lat), crs=4326)

doc = new_map(boundary, zoom=6)
add_gdf(doc, boundary, "Great Britain", opacity=0.2, symbology=[[constant("#94a3b8").encoding("fill")]])
add_gdf(doc, merge_lines(roads[roads.cls == "primary"]), "primary A-roads",
        symbology=[[constant("#059669").encoding("stroke")]])
add_gdf(doc, merge_lines(roads[roads.cls == "trunk"]), "trunk roads",
        symbology=[[constant("#64748b").encoding("stroke")]])
add_gdf(doc, merge_lines(roads[roads.cls == "motorway"]), "motorways",
        symbology=[[constant("#1d4ed8").encoding("stroke")]])
doc

## 1. Building the network

141,000 links through the consistency triggers — the same editing path as a
hand-digitised model, at national scale.

In [4]:
from aequilibrae.project import Project

SPEC = {  # free-flow speed km/h, capacity veh/h/direction
    "motorway": (105, 4000),
    "trunk": (75, 1800),
    "primary": (60, 1200),
    "ramp": (55, 1500),
}

fldr = str(Path(gettempdir()) / uuid4().hex)
project = Project()
project.new(fldr)

t0 = time.perf_counter()
with project.db_connection as conn:
    for lt, (speed, cap) in SPEC.items():
        conn.execute("insert into link_types (link_type, link_type_id, description, speed) values (?,?,?,?)",
                     (lt, lt[0], f"{lt} (UK network)", speed))
    lid = 0
    for f in roads_gj["features"]:
        cls = f["properties"]["class"]
        coords = f["geometry"]["coordinates"]
        if coords[0] == coords[-1]:
            continue  # degenerate rings cannot carry through traffic
        speed, cap = SPEC[cls]
        lid += 1
        wkt = "LINESTRING(" + ", ".join(f"{x} {y}" for x, y in coords) + ")"
        conn.execute(
            "insert into links (link_id, a_node, b_node, link_type, modes, direction, "
            " speed_ab, speed_ba, capacity_ab, capacity_ba, name, geometry) "
            "values (?, 0, 0, ?, 'c', 0, ?, ?, ?, ?, ?, GeomFromText(?, 4326))",
            (lid, cls, speed, speed, cap, cap, f["properties"]["ref"], wkt))
    conn.execute("update links set travel_time_ab = distance / 1000.0 / speed_ab * 60, "
                 "travel_time_ba = distance / 1000.0 / speed_ba * 60")
    conn.commit()
    n_nodes = conn.execute("select count(*) from nodes").fetchone()[0]
print(f"inserted {lid:,} links in {time.perf_counter() - t0:.0f}s; "
      f"triggers created {n_nodes:,} nodes")

inserted 141,075 links in 44s; triggers created 95,256 nodes


## 2. Zones

Same zone system as notebook 11: sectored megacities, spread connectors — but connectors can now attach to the much denser primary network.

In [5]:
from scipy.spatial import cKDTree

# Megacities cannot be a single point: their trips start all over the metro area,
# so cities over 1M residents are split into a centre plus ring of sector zones
# (standard practice in strategic models), each with its share of the population.
subs = []
for c in cities.itertuples():
    k = int(np.clip(round(c.population / 1_500_000) + 1, 1, 6)) if c.population >= 1_000_000 else 1
    if k == 1:
        subs.append(dict(zone=c.city, parent=c.city, lat=c.lat, lon=c.lon,
                         population=c.population, core=True))
    else:
        r = 0.10 if c.population > 5e6 else 0.05
        subs.append(dict(zone=f"{c.city} C", parent=c.city, lat=c.lat, lon=c.lon,
                         population=c.population / k, core=True))
        for i in range(k - 1):
            ang = 2 * np.pi * i / (k - 1)
            subs.append(dict(zone=f"{c.city} S{i+1}", parent=c.city,
                             lat=c.lat + r * 0.7 * np.sin(ang),
                             lon=c.lon + r * np.cos(ang) / np.cos(np.radians(54.5)),
                             population=c.population / k, core=False))
zones_df = pd.DataFrame(subs)

# Each zone gets 2-5 connectors (more for bigger zones), spread over distinct
# entry points so demand does not funnel through a single street.
nodes_df = project.network.nodes.data
xy = np.c_[nodes_df.geometry.x * np.cos(np.radians(54.5)), nodes_df.geometry.y]
tree = cKDTree(xy)

with project.db_connection as conn:
    for z in zones_df.itertuples():
        n_conn = int(np.clip(2 + z.population / 500_000, 2, 5))
        _, idx = tree.query([z.lon * np.cos(np.radians(54.5)), z.lat], k=200)
        chosen = []
        for j in np.atleast_1d(idx):
            pt = nodes_df.geometry.iloc[int(j)]
            if all(abs(pt.x - q.x) + abs(pt.y - q.y) > 0.02 for q in chosen):
                chosen.append(pt)
            if len(chosen) == n_conn:
                break
        for pt in chosen:
            lid += 1
            conn.execute(
                "insert into links (link_id, a_node, b_node, link_type, modes, direction, "
                " speed_ab, speed_ba, capacity_ab, capacity_ba, name, geometry) "
                "values (?, 0, 0, 'centroid_connector', 'c', 0, 48, 48, 10000, 10000, ?, GeomFromText(?, 4326))",
                (lid, f"{z.zone} connector", f"LINESTRING({z.lon} {z.lat}, {pt.x} {pt.y})"))
    conn.execute("update links set travel_time_ab = distance / 1000.0 / speed_ab * 60, "
                 "travel_time_ba = distance / 1000.0 / speed_ba * 60 where travel_time_ab is null")
    conn.commit()

nodes_df = project.network.nodes.data  # refresh: zone nodes now exist
key = (nodes_df.geometry.x.round(5).astype(str) + "|" + nodes_df.geometry.y.round(5).astype(str))
lookup = dict(zip(key, nodes_df.node_id))
zones_df["node_id"] = [lookup[f"{round(z.lon, 5)}|{round(z.lat, 5)}"] for z in zones_df.itertuples()]

with project.db_connection as conn:
    conn.executemany("update nodes set is_centroid = 1 where node_id = ?",
                     [(int(n),) for n in zones_df.node_id])
    conn.commit()
print(f"{len(zones_df)} zones from {cities.city.nunique()} cities "
      f"({(zones_df.groupby('parent').size() > 1).sum()} megacities sectored)")

88 zones from 74 cities (8 megacities sectored)


## 3. Skims

With primary roads available, some free-flow times shorten where the strategic network took detours.

In [6]:
from aequilibrae.paths import NetworkSkimming

project.network.build_graphs(modes=["c"])
graph = project.network.graphs["c"]
graph.set_graph("travel_time")
graph.set_skimming(["travel_time", "distance"])
graph.set_blocked_centroid_flows(True)

t0 = time.perf_counter()
skimmer = NetworkSkimming(graph)
skimmer.execute()
tt = np.array(skimmer.results.skims.get_matrix("travel_time"), copy=True)
print(f"skimmed {tt.shape[0]}x{tt.shape[1]} zone pairs in {time.perf_counter() - t0:.1f}s")

by_node = zones_df.set_index("node_id").reindex(graph.centroids)
times = pd.DataFrame(tt, index=by_node.zone, columns=by_node.zone)
rep = dict(zip(zones_df[zones_df.core].parent, zones_df[zones_df.core].zone))
pairs = [("London", "Birmingham"), ("London", "Manchester"), ("London", "Edinburgh"),
         ("Manchester", "Glasgow"), ("Bristol", "Newcastle upon Tyne"), ("Cardiff", "Norwich")]
pd.DataFrame([{"from": a, "to": b, "free-flow drive": f"{times.loc[rep[a], rep[b]] / 60:.1f} h"}
              for a, b in pairs])

                                                  :   0%|          | 0/88 [00:00<?, ?it/s]

skimmed 88x88 zone pairs in 0.2s


,from,to,free-flow drive
0,London,Birmingham,1.9 h
1,London,Manchester,3.2 h
2,London,Edinburgh,6.5 h
3,Manchester,Glasgow,3.3 h
4,Bristol,Newcastle upon Tyne,4.7 h
5,Cardiff,Norwich,4.6 h


## 4. Demand

Identical demand model and calibration to notebook 11 — so any difference in the results below comes from the network alone.

In [7]:
pop = by_node.population.to_numpy(dtype=float)
parents = by_node.parent.to_numpy()

TRIP_RATE = 0.004   # ~0.4% of residents starting a strategic inter-city car trip in the peak hour
productions = pop * TRIP_RATE
attractions = np.power(pop, 0.9)
attractions *= productions.sum() / attractions.sum()

imp = tt.copy()
np.fill_diagonal(imp, np.nan)
imp[~np.isfinite(imp)] = np.nan

T = np.outer(productions, attractions) * np.exp(-0.02 * np.nan_to_num(imp, nan=1e4))
T[np.isnan(imp)] = 0.0
np.fill_diagonal(T, 0.0)
T[parents[:, None] == parents[None, :]] = 0.0   # within-city travel is not strategic demand
for it in range(100):
    rs = T.sum(1); T *= np.divide(productions, rs, out=np.zeros_like(rs), where=rs > 0)[:, None]
    cs = T.sum(0); T *= np.divide(attractions, cs, out=np.zeros_like(cs), where=cs > 0)[None, :]
    if np.abs(T.sum(1) - productions).sum() / productions.sum() < 1e-5:
        break

lon_bir = T[np.ix_(parents == "London", parents == "Birmingham")].sum()
print(f"{T.sum():,.0f} strategic trips in the peak hour; "
      f"London -> Birmingham {lon_bir:,.0f} veh/h")

138,885 strategic trips in the peak hour; London -> Birmingham 5,663 veh/h


## 5. Equilibrium assignment

In [8]:
from aequilibrae.matrix import AequilibraeMatrix
from aequilibrae.paths import TrafficAssignment, TrafficClass

demand = AequilibraeMatrix()
demand.create_empty(zones=graph.num_zones, matrix_names=["matrix"], memory_only=True)
demand.index = graph.centroids[:]
demand.matrices[:, :, 0] = T
demand.computational_view()

assig = TrafficAssignment()
assig.add_class(TrafficClass(name="car", graph=graph, matrix=demand))
assig.set_vdf("BPR")
assig.set_vdf_parameters({"alpha": 0.15, "beta": 4.0})
assig.set_capacity_field("capacity")
assig.set_time_field("travel_time")
assig.set_algorithm("bfw")
assig.max_iter = 30
assig.rgap_target = 0.001
t0 = time.perf_counter()
assig.execute()
print(f"equilibrium in {time.perf_counter() - t0:.0f}s")

car                                               :   0%|          | 0/88 [00:00<?, ?it/s]

Equilibrium Assignment                            :   0%|          | 0/30 [00:00<?, ?it/s]

equilibrium in 8s


## 6. Two views of the result

Flow first — note how traffic that notebook 11 forced onto trunk roads now
spreads across parallel primary routes:

In [9]:
from jupytergis_lab.notebook.symbology import field

res = assig.results()
links_gdf = project.network.links.data
loaded = links_gdf.merge(res.reset_index(), on="link_id")
loaded = loaded[(loaded.matrix_tot > 100) & (loaded.link_type != "centroid_connector")].copy()
loaded["flow"] = loaded.matrix_tot.round(0)
loaded["voc"] = loaded["VOC_max"].clip(upper=1.5).round(3)
loaded["geometry"] = loaded.geometry.simplify(0.005)
fmax = float(loaded.flow.max())

doc = new_map(boundary, zoom=6)
add_gdf(doc, boundary, "Great Britain", opacity=0.2, symbology=[[constant("#94a3b8").encoding("fill")]])
add_gdf(doc, loaded[["link_id", "flow", "geometry"]], "traffic flow (veh/h)",
        symbology=[[field("flow").colormap("YlOrRd", domain=(0.0, fmax)).encoding("stroke"),
                    field("flow").scalar(domain=(100.0, fmax), output_range=(0.6, 7.0)).encoding("stroke-width")]])
doc

And congestion — the extra capacity of the primary network drains the inner-city hotspots:

In [10]:
used = loaded[loaded.flow > 0]
print(f"congestion: mean V/C {used.VOC_max.mean():.2f}, max {used.VOC_max.max():.2f}, "
      f"{(used.VOC_max > 1).mean() * 100:.1f}% of used links over capacity")

doc = new_map(boundary, zoom=6)
add_gdf(doc, boundary, "Great Britain", opacity=0.2, symbology=[[constant("#94a3b8").encoding("fill")]])
add_gdf(doc, loaded[["link_id", "voc", "flow", "geometry"]], "congestion (V/C)",
        symbology=[[field("voc").colormap("RdYlGn", reverse=True, domain=(0.0, 1.5)).encoding("stroke"),
                    field("flow").scalar(domain=(100.0, fmax), output_range=(0.6, 7.0)).encoding("stroke-width")]])
doc

congestion: mean V/C 0.29, max 2.06, 3.4% of used links over capacity


## 7. The border screenline

The same screenline as notebook 11 — now with primary-road crossings included:

In [11]:
border = LineString([(-3.6, 54.98), (-1.8, 55.82)])
crossing = loaded[loaded.geometry.intersects(border)]
print(f"{len(crossing)} links cross the border screenline; "
      f"{crossing.matrix_tot.sum():,.0f} vehicles/h in the modeled peak:")
crossing[["name", "link_type", "matrix_tot"]].sort_values("matrix_tot", ascending=False) \
        .rename(columns={"matrix_tot": "flow"}).round(0).reset_index(drop=True)

4 links cross the border screenline; 6,291 vehicles/h in the modeled peak:


,name,link_type,flow
0,A74(M),motorway,3009.0
1,A74(M),motorway,2849.0
2,A68,trunk,333.0
3,A75,trunk,100.0


## What did the detail buy?

| | Notebook 11 (strategic) | This notebook (detailed) |
|---|---|---|
| Network | 64,733 links / ~40,000 km | 141,079 links / ~70,000 km |
| Link classes | motorway, trunk, ramps | + 66,629 primary A-road links |
| Role | corridor-level flows, national screenlines | route competition, realistic diversion, urban approaches |

The build takes longer and every stage processes twice the links, but the
model's answers change where it matters: congestion spreads off the trunk
bottlenecks onto parallel primary routes, and areas the strategic network
barely reached (mid-Wales, the Borders, the South West peninsula) are now
genuinely routable. The next step down this ladder — secondary and local
roads — is where national models hand over to regional ones (notebook 09).

**Data provenance.** Road network © OpenStreetMap contributors (ODbL), Overpass
API extract (motorway/trunk/primary and ramps, topologically noded, largest
connected component), August 2026.